In [1]:
import json
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [3]:
# -----------------------------
# 1. Load dataset
# -----------------------------

with open("positive_data.json", "r") as f:
    positive_data = json.load(f)

with open("negative_data.json", "r") as f:
    negative_data = json.load(f)

data = positive_data + negative_data

df = pd.DataFrame(data)

# Convert labels
df["label"] = df["label"].map({
    "negative": 0,
    "positive": 1
})

print("Dataset size:", len(df))
print(df["label"].value_counts())


Dataset size: 196
label
1    98
0    98
Name: count, dtype: int64


In [4]:
# -----------------------------
# 2. Features and labels
# -----------------------------

X = df["sentence"]
y = df["label"]


# -----------------------------
# 3. Train-test split
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [5]:
# -----------------------------
# 4. Word + Character TF-IDF
# -----------------------------

features = FeatureUnion([

    # Word-level features
    ("word_features",
     TfidfVectorizer(
         lowercase=True,
         ngram_range=(1, 2),
         sublinear_tf=True
     )),

    # Character-level features
    ("char_features",
     TfidfVectorizer(
         analyzer="char",
         ngram_range=(3, 5),
         sublinear_tf=True
     ))
])

In [6]:
# -----------------------------
# 5. Linear SVM model
# -----------------------------

model = Pipeline([
    ("features", features),
    ("classifier", LinearSVC(C=2.0))
])


# -----------------------------
# 6. Train
# -----------------------------

model.fit(X_train, y_train)


# -----------------------------
# 7. Prediction
# -----------------------------

y_pred = model.predict(X_test)


In [7]:
# -----------------------------
# 8. Accuracy
# -----------------------------

accuracy = accuracy_score(y_test, y_pred)

print("\nAccuracy:", accuracy)
print("Accuracy percentage:", round(accuracy * 100, 2), "%")



Accuracy: 0.55
Accuracy percentage: 55.0 %


In [8]:
# -----------------------------
# 9. Classification report
# -----------------------------

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Negative", "Positive"]
    )
)


Classification Report:
              precision    recall  f1-score   support

    Negative       0.54      0.65      0.59        20
    Positive       0.56      0.45      0.50        20

    accuracy                           0.55        40
   macro avg       0.55      0.55      0.55        40
weighted avg       0.55      0.55      0.55        40



In [9]:
# -----------------------------
# 10. Confusion Matrix
# -----------------------------

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)


Confusion Matrix:
[[13  7]
 [11  9]]


In [10]:
def predict_sentiment(text):

    prediction = model.predict([text])[0]

    if prediction == 1:
        return "Positive"
    else:
        return "Negative"


# Test
test_sentences = [
    "This is an amazing product with great quality.",
    "I did not like the service at all.",
    "The staff was extremely helpful.",
    "The product is a complete waste of money.",
    "I am very happy with my purchase."
]

for text in test_sentences:
    print(
        f"{text} --> {predict_sentiment(text)}"
    )

This is an amazing product with great quality. --> Positive
I did not like the service at all. --> Negative
The staff was extremely helpful. --> Positive
The product is a complete waste of money. --> Negative
I am very happy with my purchase. --> Positive


In [11]:
import joblib

joblib.dump(model, "improved_sentiment_model.joblib")

print("Model saved successfully!")


Model saved successfully!
